# 00 · 설치 및 연결 확인
전체 프로젝트 ZIP을 업로드하고 SUMO·PyTorch 환경을 확인합니다. GPU는 필수가 아닙니다. 기존 PyTorch가 있으면 유지합니다.

In [ ]:
from pathlib import Path
import os,sys,zipfile,subprocess
if 'google.colab' in sys.modules or Path('/content').exists():
    from google.colab import files
    if not Path('/content/T_Junction_RL/REVISION_V2').exists():
        uploaded = files.upload()
        filename = next((x for x in uploaded if x.endswith('.zip')), None)
        if filename is None: raise ValueError('전체 프로젝트 ZIP을 선택하세요.')
        with zipfile.ZipFile(filename) as archive:
            for member in archive.infolist():
                destination=(Path('/content')/member.filename).resolve()
                if not destination.is_relative_to(Path('/content')): raise ValueError('잘못된 ZIP 경로')
            archive.extractall('/content')

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
subprocess.run(['bash','scripts/setup_colab.sh'], check=True)

## 선택: Drive 연결
체크포인트는 실행 중 로컬에 쓰고 완료 체크포인트와 로그를 Drive에 주기적으로 복사합니다.

In [ ]:
MOUNT_DRIVE = True
DRIVE_RUNS = None
if MOUNT_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RUNS = Path('/content/drive/MyDrive/T_Junction_RL/runs')
    DRIVE_RUNS.mkdir(parents=True,exist_ok=True)
print('지속 보관 위치:', DRIVE_RUNS)

In [ ]:
from maps.generate_map import generate
from src.traffic_generator import generate_traffic
from src.evaluate import run_episode
from src.utils import load_yaml
print('맵:', generate())
cfg=load_yaml('base.yaml');cfg['simulation']['horizon_seconds']=30
route=generate_traffic('medium',1000,30,directory=Path('outputs/setup_check/routes'))
result=run_episode(cfg,route,1000,'fixed_time',Path('outputs/setup_check/run'))
print('SUMO + TraCI simulationStep 성공:', result)